In [1]:
!pip install pymysql sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
from sqlalchemy import create_engine

In [5]:
import pandas as pd

customer = pd.read_csv('customer.csv')
address = pd.read_csv('address.csv')
city = pd.read_csv('city.csv')
country = pd.read_csv('country.csv')

payment = pd.read_csv('payment.csv')
rental = pd.read_csv('rental.csv')
inventory = pd.read_csv('inventory.csv')
staff = pd.read_csv('staff.csv')

In [7]:
# Join customer with address
dim_customer = customer.merge(
    address,
    on='address_id'
)

# Join with city
dim_customer = dim_customer.merge(
    city,
    on='city_id'
)

# Join with country
dim_customer = dim_customer.merge(
    country,
    on='country_id',
    suffixes=('_from_city', '_from_country') # Add suffixes to avoid column name conflicts
)

In [8]:
dim_customer['customer_name'] = (
    dim_customer['first_name']
    + ' ' +
    dim_customer['last_name']
)

In [9]:
dim_customer = dim_customer[[
    'customer_id',
    'customer_name',
    'email',
    'city',
    'country',
    'active'
]]

In [10]:
dim_customer.columns = [
    'customer_key',
    'customer_name',
    'email',
    'city',
    'country',
    'active_status'
]

In [11]:
dim_customer = dim_customer.drop_duplicates()

dim_customer = dim_customer.dropna()

In [12]:
dim_customer.to_csv(
    'Dim_Customer.csv',
    index=False
)

print("Dim_Customer Created!")

Dim_Customer Created!


In [15]:
fact_payment = payment.merge(
    rental,
    on='rental_id'
)

fact_payment = fact_payment.merge(
    inventory,
    on='inventory_id'
)

fact_payment = fact_payment.merge(
    staff,
    left_on='staff_id_x',
    right_on='staff_id',
    suffixes=('_from_inv', '_from_staff')
)

In [18]:
fact_payment['date_key'] = pd.to_datetime(
    fact_payment['payment_date'], format='mixed'
).dt.strftime('%Y%m%d')

In [19]:
fact_payment['payment_count'] = 1

In [22]:
fact_payment = fact_payment[[
    'payment_id',
    'customer_id_x',
    'film_id',
    'store_id_from_inv',
    'staff_id_x',
    'date_key',
    'amount',
    'payment_count'
]]

In [23]:
fact_payment.columns = [
    'payment_fact_key',
    'customer_key',
    'film_key',
    'store_key',
    'staff_key',
    'date_key',
    'payment_amount',
    'payment_count'
]

In [24]:
fact_payment = fact_payment.drop_duplicates()

fact_payment = fact_payment.dropna()

In [25]:
fact_payment.to_csv(
    'Fact_Payment.csv',
    index=False
)

print("Fact_Payment Created!")

Fact_Payment Created!
